# HW3 Part 2




**Course Code: 515512** 

**Group number: 21**

**Student Name: 游柏竣、林裕翔、楊庭愷、鄭宇豪**

**Student ID: 113550008、113550022、113550049、113550121**

# Turn Continuity Classification


**Task**: Binary classification — predict whether a spoken turn is **Complete (1)** or **Incomplete (0)**.

**Metric**: Macro-F1 Score.

| Label | Meaning |
|---|---|
| 1 | **Complete** — semantic intent is finished; system can respond |
| 0 | **Incomplete** — intent is unfinished; system should keep listening |


## 0. Environment Setup & Data Loading

In [1]:
!pip install datasets xgboost imbalanced-learn nltk transformers accelerate -q

In [2]:
import pandas as pd
import numpy as np
import random
import re
import warnings
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix
from scipy.sparse import hstack, csr_matrix

# Advanced
import xgboost as xgb
from imblearn.over_sampling import SMOTE

# NLP
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Deep Learning
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset
from torch.nn import CrossEntropyLoss

print("All imports successful!")

GPU available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
All imports successful!


In [3]:
# Load dataset (ensure train.csv and test.csv are in your current directory)
train_path = 'train.csv'
test_path = 'test.csv'

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(test_path)

# Basic exploration
print(f"Train size: {len(train_df)}")
print("Label Distribution:\n", train_df['label'].value_counts())
display(train_df.head())

Train size: 1302
Label Distribution:
 label
1    837
0    465
Name: count, dtype: int64


,id,content,label
0,1166,i think we should consider the actually the cl...,1
1,1127,Can you reset my password? My account password...,1
2,1240,im wondering if we need to no wait the test ca...,1
3,1853,"This code needs to be reviewed by tomorrow, do...",0
4,194,"This homework is taking forever, btw the new s...",0


In [4]:
# ── Standardize column names ───────────────────────────────────────────────────

# Based on your files, the text column is named 'content'
# We will normalize it to 'text' for consistency in the pipeline
text_col_mapping = {'content': 'text'}

# Rename 'content' to 'text' if it exists
train_df = train_df.rename(columns=text_col_mapping)
public_test_df = public_test_df.rename(columns=text_col_mapping)

# Ensure 'id' column exists (if not already present)
if 'id' not in train_df.columns:
    train_df['id'] = train_df.index
if 'id' not in public_test_df.columns:
    public_test_df['id'] = public_test_df.index

# ── Label Analysis (Training Set Only) ────────────────────────────────────────

print("Standardization complete.")
print(f"Final columns: {train_df.columns.tolist()}")

if 'label' in train_df.columns:
    counts = train_df['label'].value_counts()
    print("\nLabel distribution (train):")
    print(counts)

    # Calculate imbalance ratio if both classes 0 and 1 exist
    if 0 in counts and 1 in counts:
        ratio = counts[0] / counts[1]
        print(f"\nImbalance ratio (0/1): {ratio:.2f}")
    else:
        print("\nNote: Only one class found in 'label' column.")
else:
    print("\nWarning: 'label' column not found in train_df.")

# Note: public_test_df does not have a 'label' column, skipping its analysis.

Standardization complete.
Final columns: ['id', 'text', 'label']

Label distribution (train):
label
1    837
0    465
Name: count, dtype: int64

Imbalance ratio (0/1): 0.56


## Part 1: Data Balancing

You must implement and compare two methods:
1. **Basic (Required)**: Random Over-sampling.
2. **Advanced (Choose 1+)**: EDA, Back-translation, SMOTE, or Cost-Sensitive Learning.

In [5]:
# -----------------------------------------------------------------
# PHASE 1: Data Balancing
# -----------------------------------------------------------------

def perform_balancing(df, method='random'):
    """
    REQUIRED: method='random' (Random Over-sampling)
    OPTIONAL: method='advanced' (e.g., SMOTE or Cost-Sensitive hooks)
    """
    if method == 'random':
        # ── Step 1: split by class ─────────────────────────────────────────
        #TODO :Complete the function with the parameters
        #choose label 1 or 0
        df_majority = df[df['label'] == 1]
        df_minority = df[df['label'] == 0]

        # ── Step 2: upsample minority to match majority ────────────────────
        #TODO :Complete the function with the parameters
        df_minority_upsampled = resample(
            df_minority,
            replace = True,
            n_samples = len(df_majority),
            random_state = 42
        )

        # ── Step 3: combine and shuffle ────────────────────────────────────
        #TODO :Complete the function with the parameters
        df_balanced = pd.concat([df_majority, df_minority_upsampled])

        df_balanced = df_balanced.sample(
            frac=1,
            random_state=42
        )

        return df_balanced
    
    elif method == 'advanced':

        df_majority = df[df['label'] == 1]
        df_minority = df[df['label'] == 0]

        num_to_generate = len(df_majority) - len(df_minority)

        random.seed(42)

        new_samples = []

        next_id = df['id'].max() + 1

        minority_texts = df_minority['text'].tolist()

        # synonym replacement function
        def synonym_replacement(text):

            words = text.split()

            if len(words) < 3:
                return text

            idx = random.randint(0, len(words)-1)

            synonyms = wordnet.synsets(words[idx])

            if synonyms:

                synonym_words = []

                for syn in synonyms:
                    for lemma in syn.lemmas():
                        synonym_words.append(
                            lemma.name().replace('_', ' ')
                        )

                synonym_words = sorted(set(synonym_words))

                if len(synonym_words) > 0:
                    words[idx] = random.choice(synonym_words)

            return " ".join(words)

        for _ in range(num_to_generate):

            text = random.choice(minority_texts)

            augmented_text = synonym_replacement(text)

            new_samples.append({
                'id': next_id,
                'text': augmented_text,
                'label': 0
            })

            next_id += 1

        df_new = pd.DataFrame(new_samples)

        df_balanced = pd.concat([df, df_new])

        df_balanced = df_balanced.sample(
            frac=1,
            random_state=42
        ).reset_index(drop=True)

        return df_balanced

# Quick smoke-test
sample_balanced = perform_balancing(train_df, method='random') #choose your method
print("After Random Over-sampling:")
print(sample_balanced['label'].value_counts())

After Random Over-sampling:
label
0    837
1    837
Name: count, dtype: int64


## Part 2: Baseline Classifier (TF-IDF + SVM)

Establish a baseline. Use Macro-F1 as your primary metric.

In [6]:
# -----------------------------------------------------------------
# PHASE  3: Text Preprocessing (Optional)
# -----------------------------------------------------------------
"""
lemmatizer =
stop_words =
"""
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    return text

    """
    [OPTIONAL PHASE 3]: Preprocessing Function
    Hint: Use .str.lower(), or apply a lambda function for NLTK PorterStemmer/WordNetLemmatizer.
    """

"""
# Apply to entire corpus
train_df['text_clean'] = train_df['text'].apply(preprocess_text)
public_test_df['text_clean'] = public_test_df['text'].apply(preprocess_text)

print("Sample original :", train_df['text'].iloc[0])
print("Sample cleaned  :", train_df['text_clean'].iloc[0])
"""

'\n# Apply to entire corpus\ntrain_df[\'text_clean\'] = train_df[\'text\'].apply(preprocess_text)\npublic_test_df[\'text_clean\'] = public_test_df[\'text\'].apply(preprocess_text)\n\nprint("Sample original :", train_df[\'text\'].iloc[0])\nprint("Sample cleaned  :", train_df[\'text_clean\'].iloc[0])\n'

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2  ─  Model
# ══════════════════════════════════════════════════════════════════════════════
def get_model(model_type='svm'):
    """
    Model Factory for Baseline and Advanced models.
    """
    if model_type == 'svm':
        # REQUIRED PHASE 2 Baseline
        # TODO : complete the function, you could add or change the parameters
        return SVC(kernel='rbf', C=10, gamma='scale',
                   class_weight='balanced', probability=True, random_state=42)

    elif model_type == 'advanced':
        # [OPTIONAL PHASE 3]
        # Hint: Initialize xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss').
        # Try tuning max_depth, n_estimators, and learning_rate.
        return xgb.XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric='logloss', verbosity=0, random_state=42
        )


In [8]:

# --- Step 1: Internal Validation Setup ---
train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])


print(f"Train subset : {len(train_data)} samples")
print(f"Val   subset : {len(val_data)} samples")
print("Train label distribution:")
print(train_data['label'].value_counts())

Train subset : 1041 samples
Val   subset : 261 samples
Train label distribution:
label
1    669
0    372
Name: count, dtype: int64


In [9]:
# TODO: Run your Baseline experiment
# 1. Balance data
# 2. Extract features (TF-IDF)
# 3. Train SVM
# 4. Evaluate using Macro-F1 and Classification Report

print("Running Baseline Experiment...")

# Define MODEL_TYPE locally for the baseline run
# This ensures this cell can be run independently for the baseline, even if kNNUqchf_Fic hasn't been run.
MODEL_TYPE = 'svm'
BALANCE_METHOD = 'advanced'

# 1. Balance data
balanced_train_data = perform_balancing(train_data.copy(), method=BALANCE_METHOD)
print(f"Balanced train size: {len(balanced_train_data)}  |  label dist:")
print(balanced_train_data['label'].value_counts())

# 2. Extract features (TF-IDF)
# TODO : complete the function, you could add or change the parameters
vectorizer = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=50000)
#If applied text preprocessing:
#change balanced_train_data['text'] to balanced_train_data['text_clean']
X_train = vectorizer.fit_transform(balanced_train_data['text'])
y_train = balanced_train_data['label']

# 3. Train SVM
clf = get_model(MODEL_TYPE)
clf.fit(X_train, y_train)
print(f"Baseline model ({MODEL_TYPE}) trained on {X_train.shape[0]} balanced samples.")

# 4. Evaluate using Macro-F1 and Classification Report
# Prepare validation data
#If applied text preprocessing:
#change balanced_train_data['text'] to balanced_train_data['text_clean']
X_val = vectorizer.transform(val_data['text'])
y_val = val_data['label']

# Make predictions
y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')

print(f"=== Baseline: {MODEL_TYPE}| Balancing: {BALANCE_METHOD} ===")
print(classification_report(y_val, y_pred, target_names=['Incomplete(0)', 'Complete(1)']))
print(f"Macro-F1 Score: {macro_f1:.4f}")

# --- Data Balancing Comparison: random vs advanced ---
print("\n=== Data Balancing Comparison (TF-IDF + SVM) ===")
results_bal = {}
for method in ['random', 'advanced']:
    bal = perform_balancing(train_data.copy(), method=method)
    vec_bal = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=50000)
    X_tr_bal = vec_bal.fit_transform(bal['text'])
    X_va_bal = vec_bal.transform(val_data['text'])
    cw = None if method == 'random' else 'balanced'
    clf_bal = SVC(kernel='rbf', C=10, gamma='scale', class_weight=cw,
                  probability=True, random_state=42)
    clf_bal.fit(X_tr_bal, bal['label'].values)
    y_pred_bal = clf_bal.predict(X_va_bal)
    f = f1_score(val_data['label'].values, y_pred_bal, average='macro')
    results_bal[method] = f
    print(f"  {method:10s}: Macro-F1 = {f:.4f}")
    print(classification_report(val_data['label'].values, y_pred_bal,
                                target_names=['Incomplete(0)', 'Complete(1)']))


Running Baseline Experiment...
Balanced train size: 1338  |  label dist:
label
1    669
0    669
Name: count, dtype: int64
Baseline model (svm) trained on 1338 balanced samples.
=== Baseline: svm| Balancing: advanced ===
               precision    recall  f1-score   support

Incomplete(0)       0.81      0.61      0.70        93
  Complete(1)       0.81      0.92      0.86       168

     accuracy                           0.81       261
    macro avg       0.81      0.77      0.78       261
 weighted avg       0.81      0.81      0.81       261

Macro-F1 Score: 0.7814

=== Data Balancing Comparison (TF-IDF + SVM) ===
  random    : Macro-F1 = 0.7735
               precision    recall  f1-score   support

Incomplete(0)       0.87      0.56      0.68        93
  Complete(1)       0.80      0.95      0.87       168

     accuracy                           0.81       261
    macro avg       0.83      0.76      0.77       261
 weighted avg       0.82      0.81      0.80       261

  advanc

## Part 3: Enhancement with K-Fold (Optional)

Improve your results with implement  K-FOLD Cross Validtaion if needed.

In [ ]:
# --- Step 1: Training Pipeline ---

# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3  ─  Stratified K-Fold Cross-Validation
#               Tests whether the simple split was just a lucky draw.
# ══════════════════════════════════════════════════════════════════════════════

# ── Linguistic features (used in TF-IDF branch of final ensemble) ─────────────
INCOMPLETE_ENDINGS = {
    'and','but','or','because','when','if','that','the','a','an',
    'to','where','which','who','what','how','for','of','about',
    'though','although','so','yet','nor','with','in','on','at',
    'by','from','as','into','i','we','you','he','she','they',
    'it','this','these','those','my','your','his','her','our',
    'their','its','was','were','been','being','have','has','had',
    'do','does','did','will','would','could','should','may',
    'might','shall','can','not','no','both','either','just',
    'also','very','really','quite','already','still','even',
    'then','than','like','such','rather','more','most','some',
    'any','all','each','every','another','other','same','different',
}
INCOMPLETE_TRAILING = [
    ' to',' the',' a',' an',' and',' but',' or',' that',
    ' of',' in',' on',' with',' at',' by',' for',' from',
    ' as',' if',' when',' where',' which',' who',' how',
    ' been',' was',' were',' is',' are',' have',' had',
    ' will',' would',' could',' should',' may',' might',
    ' not',' no',' so',' yet',' nor',
]

def extract_linguistic_features(text):
    t = str(text).strip(); words = t.split()
    last = words[-1].lower() if words else ''; tl = t.lower(); n = len(words)
    return [
        int(last in INCOMPLETE_ENDINGS),
        int(any(tl.endswith(x) for x in INCOMPLETE_TRAILING)),
        int(t.endswith('?')), int(t.endswith('.')), int(t.endswith('!')),
        int(t.endswith(',')), int(t[-1].isalpha() if t else 0),
        int(len(last)<=2), int(len(last)<=3), int(len(last)>=7),
        int('...' in t), int(' - ' in t),
        int(any(w in tl for w in ['actually','wait ','i mean','let me','sorry,'])),
        n, len(t), 1-len(set(words))/n if n else 0,
        int(words[0].lower() in ['can','could','will','would','should','is','are',
            'was','were','do','does','did','have','has','may','might'] if words else 0),
        int(last.endswith('ing')), int(last.endswith('ed')), int(last.endswith('ly')),
        t.count(','), t.count(';'),
        int(n>=2 and words[-2].lower() in INCOMPLETE_ENDINGS),
        int(t[0].isupper() if t else 0),
        int(bool(re.search(r'[.?!][^.?!]{3,}$', t))),
    ]

def build_feature_matrix(df, vecs, fit=False):
    texts = df['text'].fillna('').tolist()
    parts = [v.fit_transform(texts) if fit else v.transform(texts) for v in vecs]
    ling  = csr_matrix(np.array([extract_linguistic_features(t) for t in texts], dtype=float))
    parts.append(ling)
    return hstack(parts)

print('Feature functions ready.')


def run_kfold_cv(df, model_type='svm', balance_method='random', n_splits=5, use_custom_preprocessing=False):
    """
    Run Stratified K-Fold CV and return fold scores + mean/std and all y_true/y_pred pairs.

    Parameters
    ----------
    df            : full training DataFrame
    model_type    : type of model to use ('svm' or 'advanced')
    balance_method: method for data balancing ('random' or 'advanced')
    n_splits      : number of folds (default 5)
    use_custom_preprocessing : bool, if True, uses the 'text_clean' column (after preprocess_text),
                                    else uses the original 'text' column.
    """
    text_col = 'text_clean' if use_custom_preprocessing and 'text_clean' in df.columns else 'text'
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_scores, all_y_true, all_y_pred = [], [], []

    for fold, (ti, vi) in enumerate(skf.split(df, df['label'])):
        tr = df.iloc[ti].copy(); va = df.iloc[vi].copy()
        tr_bal = perform_balancing(tr, method=balance_method)

        vec = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=50000)
        X_tr = vec.fit_transform(tr_bal[text_col])
        X_va = vec.transform(va[text_col])
        y_tr = tr_bal['label'].values; y_va = va['label'].values

        clf = get_model(model_type)
        clf.fit(X_tr, y_tr)
        y_pred_fold = clf.predict(X_va)
        f = f1_score(y_va, y_pred_fold, average='macro')
        print(f'  Fold {fold+1}: {f:.4f}')

        fold_scores.append(f)
        all_y_true.extend(y_va.tolist())
        all_y_pred.extend(y_pred_fold.tolist())

    mean_f1 = np.mean(fold_scores); std_f1 = np.std(fold_scores)
    print(f'\n  Mean Macro-F1: {mean_f1:.4f} +/- {std_f1:.4f}')
    return fold_scores, mean_f1, std_f1, all_y_true, all_y_pred


# ── Run Baseline K-Fold (TF-IDF + SVM) ───────────────────────────────────────
print('=== Baseline: TF-IDF + SVM (5-Fold CV) ===')
cv_scores_baseline, mean_base, std_base, _, _ = run_kfold_cv(
    train_df, model_type='svm', balance_method='random', n_splits=5
)

#Advance model

MODEL_NAME = 'distilroberta-base'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    # max_length=64: our texts avg 10 words — 64 tokens is more than enough
    return tokenizer(examples['text'], truncation=True, max_length=64, padding='max_length')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

def make_hf_dataset(df, has_labels=True):
    d  = {'text': df['text'].tolist()}
    if has_labels:
        d['labels'] = df['label'].tolist()
    ds = Dataset.from_dict(d)
    ds = ds.map(tokenize_fn, batched=True)
    ds = ds.remove_columns(['text'])
    ds.set_format('torch')
    return ds

# Custom Trainer with weighted CrossEntropyLoss for class imbalance
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(DEVICE)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = CrossEntropyLoss(weight=self.class_weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

print(f'Model: {MODEL_NAME} loaded.')

tr_df, va_df = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['label']
)
tr_bal_bert = perform_balancing(tr_df, 'random')
print(f'Train (balanced): {len(tr_bal_bert)}  |  Val: {len(va_df)}')

train_ds = make_hf_dataset(tr_bal_bert)
val_ds   = make_hf_dataset(va_df)
test_ds  = make_hf_dataset(public_test_df, has_labels=False)

# Inverse-frequency class weights
counts_bert = tr_bal_bert['label'].value_counts()
total_bert  = len(tr_bal_bert)
class_weights = torch.tensor(
    [total_bert / (2*counts_bert[0]), total_bert / (2*counts_bert[1])], dtype=torch.float
)
print(f'Class weights: {class_weights}')

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir        = './distilroberta_ckpt',
    num_train_epochs  = 15,       # more epochs, early stopping will select best
    per_device_train_batch_size = 64,   # larger batch -> faster
    per_device_eval_batch_size  = 128,
    learning_rate     = 3e-5,           # slightly higher LR for distilled model
    weight_decay      = 0.01,
    warmup_ratio      = 0.1,
    eval_strategy     = 'epoch',
    save_strategy     = 'epoch',
    load_best_model_at_end  = True,
    metric_for_best_model   = 'macro_f1',
    greater_is_better       = True,
    fp16              = torch.cuda.is_available(),
    logging_steps     = 20,
    report_to         = 'none',
    seed              = 42,
)

trainer = WeightedTrainer(
    class_weights   = class_weights,
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=4)],  # more patient
)

print('Training DistilRoBERTa...')
trainer.train()

# ── Evaluate + threshold tuning ───────────────────────────────────────────────
preds_out = trainer.predict(val_ds)
probs     = torch.softmax(torch.tensor(preds_out.predictions), dim=-1).numpy()
y_va      = va_df['label'].values

best_f1, best_t = 0.0, 0.5
for t in np.arange(0.20, 0.80, 0.005):  # finer threshold search
    f = f1_score(y_va, (probs[:,1] >= t).astype(int), average='macro')
    if f > best_f1: best_f1, best_t = f, t

distil_f1  = best_f1
distil_thr = best_t

print(f'DistilRoBERTa Val Macro-F1 : {distil_f1:.4f}  (threshold={distil_thr:.2f})')
print(classification_report(y_va, (probs[:,1] >= distil_thr).astype(int),
                             target_names=['Incomplete(0)','Complete(1)']))

print('\n=== Model Comparison ===')
print(f'  Baseline  (TF-IDF + SVM, 5-Fold CV)      : {np.mean(cv_scores_baseline):.4f}')
print(f'  Advanced  (DistilRoBERTa fine-tuned)      : {distil_f1:.4f}')
print(f'  Improvement                                : +{distil_f1 - np.mean(cv_scores_baseline):.4f}')


Feature functions ready.
=== Baseline: TF-IDF + SVM (5-Fold CV) ===
  Fold 1: 0.7908
  Fold 2: 0.7469
  Fold 3: 0.8066
  Fold 4: 0.8347
  Fold 5: 0.8048

  Mean Macro-F1: 0.7968 +/- 0.0287


Model: distilroberta-base loaded.
Train (balanced): 1338  |  Val: 261


Map:   0%|          | 0/1338 [00:00<?, ? examples/s]

Map:   0%|          | 0/261 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilRoBERTa Val Macro-F1 : 0.9576  (threshold=0.25)
               precision    recall  f1-score   support

Incomplete(0)       0.98      0.91      0.94        93
  Complete(1)       0.95      0.99      0.97       168

     accuracy                           0.96       261
    macro avg       0.97      0.95      0.96       261
 weighted avg       0.96      0.96      0.96       261


=== Model Comparison ===
  Baseline  (TF-IDF + SVM, 5-Fold CV)      : 0.7968
  Advanced  (DistilRoBERTa fine-tuned)      : 0.9576
  Improvement                                : +0.1608


## Part 4: Final Submission

Train on the full dataset using your best found configuration and generate `submission.csv`.

In [11]:
# TODO: Train final model on the entire training set

# 1. Balance data using the best method found (e.g., 'random')
# Note: This will use the entire train_df, which has 'text_clean' if preprocessing was applied.
final_balanced_train_df = perform_balancing(train_df.copy(), method='random')
print(f"Full balanced training size: {len(final_balanced_train_df)}")
print(final_balanced_train_df['label'].value_counts())

# 2. Extract features (TF-IDF) on the full balanced dataset using the preprocessed text
# Use the same TF-IDF parameters (ngram_range, stop_words) as determined during validation/K-Fold
final_vectorizer = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=50000) #TODO: fill in the parameters
# Ensure to use 'text_clean' column here for consistency with preprocessing if applied
X_full_train = final_vectorizer.fit_transform(final_balanced_train_df['text'])
y_full_train = final_balanced_train_df['label']

# 3. Train final SVM model
# The model type and hyperparameters should be chosen based on K-Fold results if K-Fold applied
final_model = get_model('svm')
final_model.fit(X_full_train, y_full_train)
print(f"Final model (svm) trained on {X_full_train.shape[0]} samples using preprocessed text.")

# ── Retrain DistilRoBERTa on full training data ───────────────────────────────
print('\nRetraining DistilRoBERTa on full data...')

train_full_bal  = perform_balancing(train_df, 'random')
train_full_ds   = make_hf_dataset(train_full_bal)

counts_f = train_full_bal['label'].value_counts(); total_f = len(train_full_bal)
cw_f = torch.tensor([total_f/(2*counts_f[0]), total_f/(2*counts_f[1])], dtype=torch.float)

model_final = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args_final = TrainingArguments(
    output_dir='./distilroberta_final',
    num_train_epochs=8,           # more epochs for full-data retrain
    per_device_train_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer_final = WeightedTrainer(
    class_weights=cw_f,
    model=model_final,
    args=args_final,
    train_dataset=train_full_ds,
)
trainer_final.train()
print('DistilRoBERTa full retrain done.')

# ── TF-IDF + LR + XGB on full data ───────────────────────────────────────────
print('Training TF-IDF + LR + XGB...')

vecs_f = [
    TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, min_df=1, max_features=80000),
    TfidfVectorizer(ngram_range=(2,5), sublinear_tf=True, min_df=1,
                    max_features=80000, analyzer='char_wb'),
]
X_tr_f = build_feature_matrix(train_full_bal, vecs_f, fit=True)
y_tr_f = train_full_bal['label'].values
X_te_f = build_feature_matrix(public_test_df, vecs_f, fit=False)

spw   = len(y_tr_f[y_tr_f==1]) / max(len(y_tr_f[y_tr_f==0]), 1)
lr_f  = LogisticRegression(C=50, class_weight='balanced', max_iter=3000, random_state=42)
xgb_f = get_model('advanced')
xgb_f.set_params(scale_pos_weight=spw, n_estimators=500, max_depth=5, learning_rate=0.05)
lr_f.fit(X_tr_f, y_tr_f)
xgb_f.fit(X_tr_f, y_tr_f)
print('TF-IDF + LR + XGB done.')


Full balanced training size: 1674
label
0    837
1    837
Name: count, dtype: int64
Final model (svm) trained on 1674 samples using preprocessed text.

Retraining DistilRoBERTa on full data...


Map:   0%|          | 0/1674 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` inste

Step,Training Loss
50,0.440525
100,0.044398
150,0.003524
200,0.001375


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilRoBERTa full retrain done.
Training TF-IDF + LR + XGB...
TF-IDF + LR + XGB done.


In [ ]:
# -----------------------------------------------------------------
# PHASE 4: Kaggle Submission
# -----------------------------------------------------------------

def generate_kaggle_submission(model, vectorizer, test_df, output_name='submission.csv'):
    """
    Generates the final CSV. Ensure test_df['text'] is processed the same way as training data.
    """
    # Ensure test data is preprocessed before vectorization, just like training data
    # If 'text_clean' exists from previous preprocessing, use it; otherwise, use 'text'
    X_test = vectorizer.transform(test_df['text'])
    test_predictions = model.predict(X_test)

    submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
    submission.to_csv(output_name, index=False)

    # Display first few rows of submission for verification
    print(f"Saved -> {output_name}")
    print("Prediction distribution:")
    print(submission['label'].value_counts())
    display(submission.head(10))

    print(f"Success: {output_name} ready for Kaggle upload.")

# ── Blend: 75% DistilRoBERTa + 25% TF-IDF ensemble ───────────────────────────
distil_prob = torch.softmax(
    torch.tensor(trainer_final.predict(test_ds).predictions), dim=-1
).numpy()
tfidf_prob  = 0.6*lr_f.predict_proba(X_te_f) + 0.4*xgb_f.predict_proba(X_te_f)
final_prob  = 0.75*distil_prob + 0.25*tfidf_prob  # higher weight to stronger model
test_preds  = (final_prob[:,1] >= distil_thr).astype(int)

print(f'Threshold: {distil_thr:.2f}')
print('Prediction dist:', pd.Series(test_preds).value_counts().to_dict())

submission = pd.DataFrame({'id': public_test_df['id'], 'label': test_preds})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
display(submission.head(10))
print('Success: submission.csv ready for Kaggle upload.')

# Final Step (SVM baseline for reference):
# generate_kaggle_submission(final_model, final_vectorizer, public_test_df)


Threshold: 0.25
Prediction dist: {1: 294, 0: 206}
Saved submission.csv


,id,label
0,1608,0
1,334,1
2,1779,1
3,1015,1
4,1790,1
5,1635,1
6,358,0
7,1574,1
8,700,1
9,284,1


Success: submission.csv ready for Kaggle upload.


In [13]:
# Download in Colab
try:
    from google.colab import files
    files.download('submission.csv')
    print("Download started!")
except ImportError:
    print("Not running in Colab — file saved locally as submission.csv")

Not running in Colab — file saved locally as submission.csv
